# BirdCLEF 2026 Inference v19 — ResNet18 + B0 + Perch 2.0 Head, 5-fold, 3-crop TTA
## Paired with training v19 | 15 models (5 folds x 3 archs)
## Speed: batched per-soundscape, parallel CPU mel, torch.inference_mode


In [ ]:
# === CELL 1: IMPORTS & CONFIG (v19) ===
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)

import os, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

try:
    import tensorflow as tf
    _TF_OK = True
except Exception:
    _TF_OK = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

import timm
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── CONFIG — must match v19 training exactly ─────────────────────────────────
CFG = dict(
    sr            = 16000,
    seconds       = 5,
    n_mels        = 64,
    n_fft         = 1024,
    hop           = 320,
    fmin          = 60,
    fmax          = 8000,
    architectures = ["resnet18", "efficientnet_b0", "perch_head"],  # v19: real Perch 2.0
    folds         = 5,
    tta_offsets   = [-1.25, 0.0, 1.25],
    device        = "cuda" if torch.cuda.is_available() else "cpu",
    perch_sr      = 32000,
    perch_emb_dim = 1536,
)

device = torch.device(CFG["device"])
torch.set_num_threads(os.cpu_count() or 4)
print("v19 inference config ready")
print(f"   Device     : {device}")
print(f"   TF OK      : {_TF_OK}")
print(f"   Models     : {len(CFG['architectures'])} archs x {CFG['folds']} folds = {len(CFG['architectures'])*CFG['folds']}")
print(f"   TTA crops  : {len(CFG['tta_offsets'])}")


In [ ]:
# === CELL 2: DATA PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV  = _first_existing(
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
TEST_AUDIO    = _first_existing(
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
)
SAMPLE_SUB    = _first_existing(
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
)
CKPT_DIR      = _first_existing(
    "/kaggle/input/birdclef-2026-v20-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species",
    "/kaggle/working",
)
# Perch 2.0 model — add "google/bird-vocalization-classifier" to notebook inputs
PERCH_MODEL_PATH = Path(_first_existing(
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/2",
    "/kaggle/input/bird-vocalization-classifier",
))
EMBD_DIR = Path("/kaggle/working/test_embs")
EMBD_DIR.mkdir(parents=True, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f"\u2705 {n_classes} species loaded")
print(f"   TEST_AUDIO       : {TEST_AUDIO}")
print(f"   CKPT_DIR         : {CKPT_DIR}")
print(f"   PERCH_MODEL_PATH : {PERCH_MODEL_PATH}  (exists={PERCH_MODEL_PATH.exists()})")
print(f"   EMBD_DIR         : {EMBD_DIR}")


In [3]:
# === CELL 3: MEL HELPER (identical params to v9 training) ===
def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("✅ logmel_from_wave defined")
print(f"   n_mels={CFG['n_mels']}, n_fft={CFG['n_fft']}, fmax={CFG['fmax']}Hz")

✅ logmel_from_wave defined
   n_mels=64, n_fft=1024, fmax=8000Hz


In [ ]:
# === CELL 4: MODEL ARCHITECTURES (pretrained=False for inference) ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.arch = arch

        if arch == "resnet18":
            self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch == "resnet50":
            self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features

        else:
            raise ValueError(f"Unsupported arch for BirdCLEFModel: {arch}")

        self.head = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))


class PerchHead(nn.Module):
    def __init__(self, n_classes: int, emb_dim: int = 1536):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


print("\u2705 BirdCLEFModel + PerchHead defined")


In [ ]:
# === CELL 4b: PERCH 2.0 EMBEDDING PRECOMPUTE (TEST SOUNDSCAPES) ===
# Extracts 1536-dim embeddings for every 5s window in each test soundscape.
# Cached to /kaggle/working/test_embs/ — safe to re-run (skips existing files).
# Requirement: add  google/bird-vocalization-classifier  to notebook inputs.

import gc

_PERCH_BATCH_INF = 16
_perch_ready_inf = False

if not _TF_OK:
    print("TensorFlow not available — perch_head will output neutral 0.5")
elif not PERCH_MODEL_PATH.exists():
    print(f"WARNING: Perch model not found at {PERCH_MODEL_PATH}")
    print("Add 'google/bird-vocalization-classifier' (TF2 > perch_v2 > v2) to this notebook's inputs.")
else:
    _existing_embs = len(list(EMBD_DIR.glob("*.npy")))
    _sample_sub_df = pd.read_csv(SAMPLE_SUB)
    _total_rows    = len(_sample_sub_df)

    if _existing_embs >= _total_rows:
        print(f"Reusing {_existing_embs} cached test embeddings")
        _perch_ready_inf = True
    else:
        print(f"Loading Perch 2.0 from {PERCH_MODEL_PATH} ...")
        _perch_model  = tf.saved_model.load(str(PERCH_MODEL_PATH))
        _perch_infer  = _perch_model.signatures["serving_default"]
        _perch_target = CFG["perch_sr"] * CFG["seconds"]   # 160 000 samples

        _audio_batch, _name_batch = [], []

        def _flush_batch_inf():
            _audio_tf = tf.constant(np.stack(_audio_batch), dtype=tf.float32)
            _out  = _perch_infer(inputs=_audio_tf)
            _embs = _out["embedding"].numpy()
            for _n, _e in zip(_name_batch, _embs):
                np.save(str(EMBD_DIR / (_n + ".npy")), _e.astype(np.float32))
            _audio_batch.clear()
            _name_batch.clear()

        # Group rows by soundscape — load each .ogg once
        _rows_by_sc = defaultdict(list)
        for _rid in _sample_sub_df["row_id"]:
            _sc, _es = _rid.rsplit("_", 1)
            _rows_by_sc[_sc].append(int(_es))

        _failed = 0
        for _sc_id, _end_secs in tqdm(_rows_by_sc.items(), desc="Perch embed (test)"):
            _apath = Path(TEST_AUDIO) / f"{_sc_id}.ogg"
            if not _apath.exists():
                for _ext in (".wav", ".flac"):
                    _fb = _apath.with_suffix(_ext)
                    if _fb.exists():
                        _apath = _fb
                        break
                else:
                    _failed += 1
                    continue
            try:
                _y, _sr0 = sf.read(str(_apath), always_2d=False)
                if _y.ndim == 2:
                    _y = _y.mean(axis=1)
                if _sr0 != CFG["perch_sr"]:
                    _y = librosa.resample(
                        _y.astype(np.float32), orig_sr=_sr0, target_sr=CFG["perch_sr"]
                    )
                _y = _y.astype(np.float32)
                for _es in _end_secs:
                    _emb_name = f"{_sc_id}_{_es}"
                    if (EMBD_DIR / (_emb_name + ".npy")).exists():
                        continue
                    _end_samp   = int(_es * CFG["perch_sr"])
                    _start_samp = max(0, _end_samp - _perch_target)
                    _clip = _y[_start_samp:_end_samp]
                    if len(_clip) < _perch_target:
                        _clip = np.pad(_clip, (0, _perch_target - len(_clip)))
                    _audio_batch.append(_clip)
                    _name_batch.append(_emb_name)
                    if len(_audio_batch) >= _PERCH_BATCH_INF:
                        _flush_batch_inf()
            except Exception:
                _failed += 1

        if _audio_batch:
            _flush_batch_inf()

        del _perch_model, _perch_infer
        tf.keras.backend.clear_session()
        gc.collect()

        _saved = len(list(EMBD_DIR.glob("*.npy")))
        print(f"Test Perch embeddings: {_saved} saved  ({_failed} soundscapes failed)")
        print("TF model freed — ready for PyTorch inference")
        _perch_ready_inf = _saved > 0


In [ ]:
# === CELL 5: LOAD 15 CHECKPOINTS (5 folds x 3 architectures) ===
mel_models = []   # BirdCLEFModel instances
emb_models = []   # PerchHead instances
missing    = []

for arch in CFG["architectures"]:
    for fold_idx in range(CFG["folds"]):
        ckpt_path = Path(CKPT_DIR) / f"{arch}_v20_fold{fold_idx}.pt"
        if not ckpt_path.exists():
            missing.append(str(ckpt_path))
            continue
        if arch == "perch_head":
            m = PerchHead(n_classes=n_classes, emb_dim=CFG["perch_emb_dim"]).to(device)
        else:
            m = BirdCLEFModel(arch, n_classes=n_classes, pretrained=False).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
        m.eval()
        if arch == "perch_head":
            emb_models.append(m)
        else:
            mel_models.append(m)
        print(f"   \u2705 Loaded {ckpt_path.name}")

models = mel_models + emb_models

if missing:
    print(f"\n\u26a0\ufe0f  {len(missing)} checkpoint(s) not found:")
    for p in missing:
        print(f"      {p}")

print(f"\n\u2705 {len(mel_models)} mel models + {len(emb_models)} embedding models")
print(f"   Total: {len(models)}/{len(CFG['architectures'])*CFG['folds']} models")


In [ ]:
# === CELL 6: BATCHED PREDICTION PER SOUNDSCAPE ===
_use_amp    = (device.type == "cuda")
win_samples = CFG["sr"] * CFG["seconds"]
win_frames  = int(CFG["seconds"] * CFG["sr"] / CFG["hop"]) + 1
_n_tta      = len(CFG["tta_offsets"])
_pad_samps  = win_samples + int(max(abs(o) for o in CFG["tta_offsets"]) * CFG["sr"]) + 1
_CHUNK      = 512
_N_WORKERS  = (2 if device.type == "cpu" else min(4, (os.cpu_count() or 1)))


def _extract_mel_crop(args):
    wave_padded, nom_start, offset_s = args
    off   = int(offset_s * CFG["sr"])
    start = max(0, nom_start + off)
    end_  = start + win_samples
    if end_ > len(wave_padded):
        end_  = len(wave_padded)
        start = max(0, end_ - win_samples)
    clip = wave_padded[start:end_]
    if len(clip) < win_samples:
        clip = np.pad(clip, (0, win_samples - len(clip)))
    mel = logmel_from_wave(clip, CFG["sr"])
    T = mel.shape[1]
    if T < win_frames:
        mel = np.concatenate(
            [mel, np.zeros((mel.shape[0], win_frames - T), dtype=np.float32)], axis=1
        )
    elif T > win_frames:
        mel = mel[:, :win_frames]
    return mel


def predict_soundscape(audio_path: str, end_seconds, soundscape_id: str = None) -> np.ndarray:
    end_seconds = list(end_seconds)
    n_rows      = len(end_seconds)

    if not models:
        return np.full((n_rows, n_classes), 0.5, dtype=np.float32)

    all_model_probs = []

    # MEL-based models (3-crop TTA)
    if mel_models:
        _wave = None
        try:
            _wave, _sr0 = sf.read(audio_path, always_2d=False)
        except Exception:
            pass

        if _wave is not None:
            if _sr0 != CFG["sr"]:
                _wave = librosa.resample(_wave, orig_sr=_sr0, target_sr=CFG["sr"])
            if _wave.ndim == 2:
                _wave = _wave.mean(axis=1)
            _wave = np.pad(_wave.astype(np.float32), (_pad_samps, _pad_samps))

            tasks = [
                (_wave, int(end_s * CFG["sr"]) + _pad_samps - win_samples, off)
                for end_s in end_seconds
                for off in CFG["tta_offsets"]
            ]
            with ThreadPoolExecutor(max_workers=_N_WORKERS) as pool:
                mels = list(pool.map(_extract_mel_crop, tasks))

            batch = torch.from_numpy(np.stack(mels)[:, None]).float().to(device)
            for m in mel_models:
                chunks = []
                for i in range(0, len(batch), _CHUNK):
                    with torch.inference_mode(), autocast(enabled=_use_amp):
                        p = torch.sigmoid(m(batch[i:i + _CHUNK])).float().cpu().numpy()
                    chunks.append(p)
                probs = np.concatenate(chunks, axis=0)
                probs = probs.reshape(n_rows, _n_tta, n_classes).mean(axis=1)
                all_model_probs.append(probs)
        else:
            neutral = np.full((n_rows, n_classes), 0.5, dtype=np.float32)
            all_model_probs.extend([neutral] * len(mel_models))

    # Perch embedding models
    if emb_models:
        if _perch_ready_inf and soundscape_id is not None:
            _emb_rows = []
            for end_s in end_seconds:
                _ep = EMBD_DIR / f"{soundscape_id}_{end_s}.npy"
                _emb_rows.append(
                    np.load(str(_ep)) if _ep.exists()
                    else np.zeros(CFG["perch_emb_dim"], dtype=np.float32)
                )
            emb_batch = torch.from_numpy(np.stack(_emb_rows)).float().to(device)
            for m in emb_models:
                with torch.inference_mode(), autocast(enabled=_use_amp):
                    p = torch.sigmoid(m(emb_batch)).float().cpu().numpy()
                all_model_probs.append(p)
        else:
            neutral = np.full((n_rows, n_classes), 0.5, dtype=np.float32)
            all_model_probs.extend([neutral] * len(emb_models))

    if not all_model_probs:
        return np.full((n_rows, n_classes), 0.5, dtype=np.float32)

    return np.mean(all_model_probs, axis=0).astype(np.float32)


_status = f"{len(mel_models)} mel + {len(emb_models)} emb models" if models else "WARNING: 0 models"
print("predict_soundscape() ready — mel (3-crop TTA) + Perch embeddings")
print(f"   CPU mel workers : {_N_WORKERS}")
print(f"   Models          : {_status}")


In [ ]:
# === CELL 7: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB)
print(f"Sample submission rows: {len(sample_sub)}")

all_row_ids    = []
all_probs_list = []
missing_audio  = 0

sample_sub = sample_sub.copy()
sample_sub["_soundscape_id"] = sample_sub["row_id"].str.rsplit("_", n=1).str[0]

for soundscape_id, group in tqdm(
    sample_sub.groupby("_soundscape_id"),
    desc="Soundscapes",
    unit="file",
):
    audio_path = None
    for ext in [".ogg", ".wav", ".flac"]:
        cand = Path(TEST_AUDIO) / f"{soundscape_id}{ext}"
        if cand.exists():
            audio_path = str(cand)
            break

    row_ids = [str(r) for r in group["row_id"]]

    if audio_path is None:
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    end_seconds = [int(rid.rsplit("_", 1)[-1]) for rid in row_ids]
    probs_all   = predict_soundscape(audio_path, end_seconds, soundscape_id=soundscape_id)

    all_row_ids.extend(row_ids)
    all_probs_list.append(probs_all)

if missing_audio:
    print(f"\u26a0\ufe0f  {missing_audio} soundscape(s) not found in {TEST_AUDIO} — used neutral 0.5")
print(f"\n\u2705 Generated {len(all_row_ids)} prediction rows")


In [8]:
# === CELL 8: BUILD SUBMISSION ===
submission_df = sample_sub[["row_id"]].copy()
submission_df["row_id"] = submission_df["row_id"].astype(str)

if all_probs_list:
    # Stack into (N, n_classes) and build DataFrame in one shot — much faster than list-of-dicts
    probs_np = np.vstack(all_probs_list).astype(np.float64)   # (N, n_classes)
    pred_df  = pd.DataFrame(probs_np, columns=species)
    pred_df.insert(0, "row_id", all_row_ids)
    pred_df["row_id"] = pred_df["row_id"].astype(str)
    submission_df = submission_df.merge(pred_df, on="row_id", how="left")
else:
    print("⚠️  No predictions — building fully neutral submission (0.5)")

# Fill any missing species columns (left-join NaN or fully absent)
for sp in species:
    if sp in submission_df.columns:
        submission_df[sp] = submission_df[sp].fillna(0.5).astype(np.float64)
    else:
        submission_df[sp] = 0.5

submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print(f"✅ Saved submission.csv  ({len(submission_df)} rows × {len(submission_df.columns)} columns)")
submission_df.head(3)


✅ Saved submission.csv  (3 rows × 235 columns)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,BC2026_Test_0001_S05_20250227_010002_10,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,BC2026_Test_0001_S05_20250227_010002_15,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [9]:
# === CELL 9: FORMAT VALIDATION ===
print("Running submission validation...")
issues = []

# Check row_id dtype
if submission_df["row_id"].dtype != object:
    issues.append(f"row_id dtype is {submission_df['row_id'].dtype}, expected str")

# Check species columns exist and are numeric
missing_cols = [sp for sp in species if sp not in submission_df.columns]
if missing_cols:
    issues.append(f"{len(missing_cols)} species columns missing: {missing_cols[:5]}")

# Check value range
sp_cols  = [c for c in submission_df.columns if c != "row_id"]
val_min  = submission_df[sp_cols].min().min()
val_max  = submission_df[sp_cols].max().max()
if val_min < 0 or val_max > 1:
    issues.append(f"Predictions out of [0,1]: min={val_min:.4f} max={val_max:.4f}")

# Check for NaN
nan_count = submission_df[sp_cols].isna().sum().sum()
if nan_count > 0:
    issues.append(f"{nan_count} NaN values in species columns")

if issues:
    print("\n⚠️  Issues found:")
    for issue in issues:
        print(f"   • {issue}")
else:
    print("✅ Submission looks valid!")
    print(f"   Rows        : {len(submission_df)}")
    print(f"   Species cols: {len(sp_cols)}")
    print(f"   Value range : [{val_min:.4f}, {val_max:.4f}]")
    print(f"   NaN count   : {nan_count}")

Running submission validation...
✅ Submission looks valid!
   Rows        : 3
   Species cols: 234
   Value range : [0.5000, 0.5000]
   NaN count   : 0
